# Notebook 2 — Primitive Sectors and Transport Topology

The six spectral layers are further refined by the commutative center $\mathrm{Center}\{A, \mathrm{QT}_{\mathrm{all}}, \mathrm{HT}_{\mathrm{all}}\}$. Joint diagonalization yields **9 primitive sectors** — indivisible spectral units. We compute the transport tensor $K_{ij}$ between them and reveal the **star topology**.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from rime.cubieoperator import CubieSpectralOperator, eigenspaces
from rime.cubie import CubieMove
from rime.cubieworld import SlowDynamics
from rime import helpers
from rime.spectral_utils import joint_diag_sectors, compute_transport_kappa
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
op = CubieSpectralOperator.from_gens_dict(CubieMove.prim_moves)
layers = sorted(op._layers, reverse=True)
print(f'Operator built: {len(layers)} spectral layers, dims={[op._layer_dim(lam) for lam in layers]}')

## 2. The Center and Joint Diagonalization

The three operators $\{A_{18}, \mathrm{QT}_{\mathrm{all}}, \mathrm{HT}_{\mathrm{all}}\}$ commute pairwise. Their joint diagonalization yields the finest decomposition where all three are simultaneously diagonal.

In [ ]:
# Build QT_all and HT_all from per-axis operators
QT_ops = op.build_per_axis_ops()  # (QT^0, QT^1, QT^2) per-axis quarter-turn averages
HT_ops = op.build_per_axis_ops(half_turn=True)  # (HT^0, HT^1, HT^2) per-axis half-turn averages
QT_all = sum(QT_ops)
HT_all = sum(HT_ops)

A = op._build_A()
print(f'||[A, QT_all]|| = {np.linalg.norm(A @ QT_all - QT_all @ A):.1e}')
print(f'||[A, HT_all]|| = {np.linalg.norm(A @ HT_all - HT_all @ A):.1e}')
print(f'||[QT_all, HT_all]|| = {np.linalg.norm(QT_all @ HT_all - HT_all @ QT_all):.1e}')
print('All three commute — joint diagonalization is valid.')

# Joint diagonalization
ops_dict = {'A': A, 'QT': QT_all, 'HT': HT_all}
sectors = joint_diag_sectors(ops_dict, tol=1e-10)
print(f'\n{sectors["n_sectors"]} primitive sectors found')
for i in range(sectors['n_sectors']):
    dim = sectors['dims'][i]
    lam_A = sectors['eigenvalues'][i, 0]
    lam_QT = sectors['eigenvalues'][i, 1]
    lam_HT = sectors['eigenvalues'][i, 2]
    print(f'  S{i+1}: dim={dim:>3d}  λ_A={lam_A:.4f}  λ_QT={lam_QT:.4f}  λ_HT={lam_HT:.4f}')

## 3. Sector Block Support

Each sector lives in a specific combination of cubie-type blocks. **Hybrid sectors** (S6, S7) span multiple blocks — they are the transport hubs.

In [ ]:
block_slices = {'cp': slice(0,64), 'ep': slice(64,208), 'co': slice(208,216), 'eo': slice(216,228)}

for i in range(sectors['n_sectors']):
    P = sectors['projectors'][i]
    blocks = []
    for bn, sl in block_slices.items():
        w = np.linalg.norm(P[sl][:, sl], 'fro')**2
        if w > 1e-8:
            blocks.append(f'{bn}({w:.1f})')
    is_hybrid = '★' if len(blocks) > 1 else ' '
    print(f'  S{i+1} {is_hybrid} {", ".join(blocks)}')

## 4. Transport Tensor $K_{ij}$

$K_{ij} = \max_g \|P_i \rho(g) P_j\|_F$ — the maximum Frobenius norm of the inter-sector block of each generator. Nonzero $K_{ij}$ means a single generator move can transfer amplitude from sector $j$ to sector $i$.

In [ ]:
n = sectors['n_sectors']
K = np.zeros((n, n))
for g_idx, (_, rho, *_) in enumerate(op.rho_moves.values()):
    rho_dense = rho.toarray() if hasattr(rho, 'toarray') else np.array(rho)
    for i in range(n):
        Pi_rho = sectors['projectors'][i] @ rho_dense
        for j in range(n):
            val = np.linalg.norm(Pi_rho @ sectors['projectors'][j], 'fro')
            K[i, j] = max(K[i, j], val)

nz = int(np.sum(K > 1e-8))
print(f'{nz} nonzero connections out of {n*n} possible directed pairs')

# Show K matrix
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(K, cmap='inferno')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([f'S{i+1}' for i in range(n)], fontsize=8)
ax.set_yticklabels([f'S{i+1}' for i in range(n)], fontsize=8)
ax.set_title(f'Transport Tensor $K_{{ij}}$ ({nz} nonzero)', fontsize=12)
plt.colorbar(im, ax=ax, label='K')
plt.tight_layout(); plt.show()

## 5. Transport Skeleton

The nonzero K edges form a **star topology**: S6 is the primary hub (degree 5), S7 is the secondary hub (degree 3). S1 (V₁, the solved-state component) is **fully isolated** — no direct transport to or from any other sector.

In [ ]:
import networkx as nx

G = nx.DiGraph()
for i in range(n):
    G.add_node(i, label=f'S{i+1}')
for i in range(n):
    for j in range(n):
        if K[i, j] > 0.05:
            G.add_edge(j, i, weight=K[i, j])

pos = nx.spring_layout(G, seed=42, k=2)
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
degrees = dict(G.degree())
sizes = [300 + 200 * degrees.get(i, 0) for i in range(n)]
colors_node = ['#e94560' if degrees.get(i, 0) >= 4 else '#0f3460' if degrees.get(i, 0) > 0 else '#666666' for i in range(n)]

nx.draw_networkx_nodes(G, pos, ax=ax, node_size=sizes, node_color=colors_node)
nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#888888', width=1.5,
                       arrowstyle='-|>', arrowsize=15, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G, pos, ax=ax, font_size=10, font_color='white', font_weight='bold')
ax.set_title('Transport Skeleton: Star Topology', fontsize=14)
ax.axis('off')
plt.tight_layout(); plt.show()
for i in range(n):
    d = degrees.get(i, 0)
    hub = ' ← HUB' if d >= 4 else ' (isolated)' if d == 0 else ''
    print(f'  S{i+1}: degree = {d}{hub}')

## 6. T7 Pairs — Composition-Only Transport

Some sector pairs have $K=0$ (no direct transport) but are reachable via 2-step composition through a hybrid hub. At the Lie (continuous) level, these pairs also have $\kappa_0=\kappa_1=0$ — the continuous limit **annihilates** these channels. This is the T7 mechanism.

In [ ]:
# Find T7 candidates: K=0 but reachable via composition
K_binary = (K > 1e-8).astype(int)
K2 = (K_binary @ K_binary) > 0  # 2-step reachable

t7_pairs = []
for i in range(n):
    for j in range(n):
        if i != j and K_binary[i, j] == 0 and K2[i, j]:
            t7_pairs.append((i, j))

print(f'{len(t7_pairs)} composition-only pairs (K=0, reachable in 2 steps):')
for i, j in t7_pairs:
    # Find the mediating hub
    mediators = [k for k in range(n) if K_binary[i, k] and K_binary[k, j]]
    print(f'  S{i+1} → S{j+1}  via {[f"S{k+1}" for k in mediators]}')